In [1]:
from dataclasses import dataclass
from training import count_parameters, TrainConfig, train_waitk_student, TranslationDataset, save_and_log_checkpoint, load_training_checkpoint
import mlflow
import torch
import json
from mamba_ssm import Mamba2
from evaluation import SimulMTEvaluator, MTQualityScorer, WaitKMamba2Adapter
from model_classes import WaitKMamba2MT, SimulMamba2Config, WaitKTransformerMT, SimulTransformerConfig

In [2]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("simulmt_waitk_mamba2_distillation")

<Experiment: artifact_location='/mlflow/artifacts/2', creation_time=1779451055489, experiment_id='2', last_update_time=1779451055489, lifecycle_stage='active', name='simulmt_waitk_mamba2_distillation', tags={}, trace_location=None, workspace='default'>

In [3]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

teacher_name = "./models/nllb_teacher"
tokenizer = AutoTokenizer.from_pretrained(teacher_name)

In [4]:
mamba2_config = SimulMamba2Config(
    vocab_size=len(tokenizer),

    d_model=512,
    num_layers=12,

    d_state=128,
    d_conv=4,
    expand=2,
    headdim=64,
    ngroups=1,

    dropout=0.1,

    max_source_len=64,
    max_target_len=64,

    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,

    tie_embeddings=True,
    separate_source_target_embeddings=True,
)

mamba2 = WaitKMamba2MT(mamba2_config)
count_parameters(mamba2)

Total parameters:     283,070,528
Trainable parameters: 283,070,528
Frozen parameters:    0


{'total': 283070528, 'trainable': 283070528, 'frozen': 0}

In [5]:
dataset = TranslationDataset("./data/train_dataset.hdf5")

In [6]:
train_cfg = TrainConfig(
    epochs=8,
    short_epochs=False,
    batches_per_epoch=2000,
    batch_size=192,
    gradient_accumulation_steps=8,

    wait_k=[3, 5, 7, 9, 11],

    use_kl_loss=False,
    use_dataset_ce_loss=True,

    kl_weight=1.0,
    dataset_ce_weight=1.0,

    lr=1e-4,
    use_amp=True,
    save_every_steps=100000
)

In [7]:
train_waitk_student(
    student=mamba2,
    train_dataset=dataset,
    model_cfg=mamba2_config,
    train_cfg=train_cfg,
    device="cuda",
    checkpoint_name_prefix="mamba",
    compile_model=False,
    save_latest=False
)

epoch 1/8:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 2/8:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 3/8:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 4/8:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 5/8:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 6/8:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 7/8:   0%|          | 0/14504 [00:00<?, ?it/s]

epoch 8/8:   0%|          | 0/14504 [00:00<?, ?it/s]

🏃 View run waitk-student at: http://localhost:5000/#/experiments/2/runs/e3576becbb304f528584daf0fd3c7e1d
🧪 View experiment at: http://localhost:5000/#/experiments/2


WaitKMamba2MT(
  (src_embedding): Embedding(256204, 512, padding_idx=1)
  (tgt_embedding): Embedding(256204, 512, padding_idx=1)
  (position_embedding): Embedding(132, 512)
  (segment_embedding): Embedding(3, 512)
  (dropout): Dropout(p=0.1, inplace=False)
  (layers): ModuleList(
    (0-11): 12 x Mamba2Block(
      (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (mixer): Mamba2(
        (in_proj): Linear(in_features=512, out_features=2320, bias=False)
        (conv1d): Conv1d(1280, 1280, kernel_size=(4,), stride=(1,), padding=(3,), groups=1280)
        (act): SiLU()
        (norm): RMSNorm()
        (out_proj): Linear(in_features=1024, out_features=512, bias=False)
      )
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (final_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=512, out_features=256204, bias=False)
)

In [8]:
eval_dataset = TranslationDataset(
    "./data/val_dataset.hdf5",
    lazy=False,
)

In [9]:
adapter = WaitKMamba2Adapter(
    model=mamba2,
    tokenizer=tokenizer,
    name="mamba2_wait_k",
    device="cuda",
    use_amp=True,
)

evaluator = SimulMTEvaluator(
    tokenizer=tokenizer,
    quality_scorer=MTQualityScorer(),
)

for k in [3, 5, 7, 9, 11]:
    result = evaluator.evaluate(
        model=adapter,
        dataset=eval_dataset,
        wait_k=k,
        batch_size=1024,
        max_new_tokens=63
    )
    
    print(result.metrics)
    
    with open(f"./metrics/mamba_k{k}.json", "w") as file:
        json.dump(result.metrics, file)

Validating mamba2_wait_k, wait_k=3:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 26.69841907135322, 'chrF++': 49.09545752357134, 'TER': 68.33847924428852, 'AP': 0.6632980618526004, 'AL': 4.470308448637609, 'DAL': 5.211351467785213, 'LAAL': 4.7977169106113795, 'ATD_text': 3.136225085008017, 'total_time_sec': 483.45188825498917, 'ms_per_sentence': 1.562496002892567, 'target_tokens_per_sec': 18456.798322181163, 'source_tokens_per_sec': 16099.35174334212, 'first_token_latency_sec': 1.4555163742414392, 'peak_gpu_memory_mb': 5630.20361328125, 'generation_total_time_sec': 440.48423591403116, 'generation_ms_per_sentence': 1.4236263724961415, 'generation_target_tokens_per_sec': 20257.192590523235}


Validating mamba2_wait_k, wait_k=5:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 28.037714791686778, 'chrF++': 49.831935168938344, 'TER': 65.99632008936234, 'AP': 0.7287203675133733, 'AL': 6.318419506579318, 'DAL': 7.0838540553726155, 'LAAL': 6.214181891433729, 'ATD_text': 4.392455154677043, 'total_time_sec': 484.9009285840002, 'ms_per_sentence': 1.5671792397918627, 'target_tokens_per_sec': 18168.78145753812, 'source_tokens_per_sec': 16051.241689160206, 'first_token_latency_sec': 1.457412943174074, 'peak_gpu_memory_mb': 5630.20361328125, 'generation_total_time_sec': 441.06289165696944, 'generation_ms_per_sentence': 1.4254965633204144, 'generation_target_tokens_per_sec': 19974.609441530396}


Validating mamba2_wait_k, wait_k=7:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 28.085253275459753, 'chrF++': 49.56050000334846, 'TER': 65.82692814503007, 'AP': 0.7839712047551884, 'AL': 8.13737410957781, 'DAL': 8.908922184518717, 'LAAL': 7.484698845949543, 'ATD_text': 5.577577308784585, 'total_time_sec': 489.41860635699413, 'ms_per_sentence': 1.5817801827898068, 'target_tokens_per_sec': 17895.455722848892, 'source_tokens_per_sec': 15903.077445164998, 'first_token_latency_sec': 1.4613928396313807, 'peak_gpu_memory_mb': 5630.20361328125, 'generation_total_time_sec': 442.25022030301625, 'generation_ms_per_sentence': 1.4293339591578043, 'generation_target_tokens_per_sec': 19804.103192982097}


Validating mamba2_wait_k, wait_k=9:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 27.58208679744893, 'chrF++': 48.950269590266885, 'TER': 66.35169017183394, 'AP': 0.8286708569453043, 'AL': 9.897052719491766, 'DAL': 10.655608060269518, 'LAAL': 8.594299219026185, 'ATD_text': 6.632959032196819, 'total_time_sec': 489.0630717870081, 'ms_per_sentence': 1.580631110135445, 'target_tokens_per_sec': 17849.89197426438, 'source_tokens_per_sec': 15914.63851801448, 'first_token_latency_sec': 1.4596040548418805, 'peak_gpu_memory_mb': 5630.16455078125, 'generation_total_time_sec': 441.72392659705656, 'generation_ms_per_sentence': 1.427633000216724, 'generation_target_tokens_per_sec': 19762.84840907726}


Validating mamba2_wait_k, wait_k=11:   0%|          | 0/303 [00:00<?, ?it/s]

{'BLEU': 26.882324669371854, 'chrF++': 48.1733010149638, 'TER': 67.16992628089662, 'AP': 0.8643678814959429, 'AL': 11.574227183447197, 'DAL': 12.302291953411098, 'LAAL': 9.555877195099969, 'ATD_text': 7.546617793349301, 'total_time_sec': 489.6663116560085, 'ms_per_sentence': 1.5825807558127032, 'target_tokens_per_sec': 17777.573814625284, 'source_tokens_per_sec': 15895.032626765136, 'first_token_latency_sec': 1.4599166188812, 'peak_gpu_memory_mb': 5626.43212890625, 'generation_total_time_sec': 441.81883310693956, 'generation_ms_per_sentence': 1.427939734032318, 'generation_target_tokens_per_sec': 19702.82465956581}
